# MachineGuard — Baseline Machine Learning

This notebook develops and evaluates supervised baseline models for bearing-condition classification using the engineered vibration features from NASA IMS Set 1, Bearing 4 X-axis.

The initial prediction task is binary classification:

- **0 — NORMAL**
- **1 — DEGRADED**

The condition labels are heuristic and are derived from the run-to-failure timeline rather than from ground-truth labels provided by NASA.

The main goals are to:

- define a time-aware training and evaluation strategy,
- train a simple supervised baseline,
- train a stronger model,
- compare their performance,
- avoid temporal leakage,
- determine which model is most appropriate for the first MachineGuard prediction pipeline.

In [86]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [87]:
file_path = Path(
    "../data/processed/ims_set1_bearing4_x_features.csv"
)

features_df = pd.read_csv(
    file_path,
    parse_dates=["timestamp"],
)

In [88]:
features_df.head()

,rms,std,peak,crest_factor,kurtosis,dominant_frequency,spectral_energy,timestamp
0,0.109020,0.060085,0.413,3.788279,3.790250,986.328125,757118.612018,2003-10-22 12:06:24
1,0.108900,0.059420,0.420,3.856767,4.234703,986.328125,740448.654400,2003-10-22 12:09:13
2,0.110319,0.059968,0.396,3.589583,3.924708,986.328125,754376.681760,2003-10-22 12:14:13
3,0.111037,0.059915,0.393,3.539356,3.652667,986.328125,752875.194856,2003-10-22 12:19:13
4,0.110190,0.059255,0.376,3.412278,3.635576,986.328125,736347.011716,2003-10-22 12:24:13


In [89]:
features_df.shape

(2156, 8)

## Heuristic condition labels

The NASA IMS dataset does not provide a NORMAL/DEGRADED label for every snapshot.

For the initial supervised baseline, a heuristic degradation boundary is defined from the run-to-failure timeline.

Snapshots recorded before November 20, 2003 are labeled as NORMAL, while snapshots recorded from November 20 onward are labeled as DEGRADED.

These labels are used only as an approximate condition target and should not be interpreted as ground-truth mechanical failure labels.

In [90]:
degradation_start = pd.Timestamp("2003-11-20")

features_df["target"] = (
    features_df["timestamp"] >= degradation_start
).astype(int)

In [91]:
features_df["target"].value_counts()

target
0    1507
1     649
Name: count, dtype: int64

## Time-aware train/test strategy

Because the NASA IMS data form a chronological run-to-failure sequence, a random train/test split would risk temporal leakage.

The supervised models should therefore be trained on earlier observations and evaluated on later observations.

However, the training set must contain examples of both NORMAL and DEGRADED conditions. The split point must therefore occur after the heuristic degradation boundary while still leaving a later degraded period for testing.

In [92]:
features_df["timestamp"].min(), features_df["timestamp"].max()

(Timestamp('2003-10-22 12:06:24'), Timestamp('2003-11-25 23:39:56'))

In [93]:
features_df.groupby(
    features_df["timestamp"].dt.date
)["target"].agg(["count", "sum"])

,count,sum
timestamp,,
2003-10-22,94,0
2003-10-23,62,0
2003-10-29,57,0
2003-10-30,22,0
2003-10-31,133,0
2003-11-01,144,0
2003-11-03,28,0
2003-11-07,48,0
2003-11-08,82,0


## Chronological holdout split

A strict chronological split is used to reduce temporal leakage between neighboring vibration snapshots.

The model is trained using observations up to November 22, 2003, and evaluated on the final three days of the experiment.

This allows the model to learn from both NORMAL observations and the early DEGRADED period while reserving the latest stage of degradation for testing.

Because the heuristic target is defined directly from time, the final test period contains only DEGRADED observations. Therefore, this holdout is mainly used to evaluate whether the model generalizes to later degradation rather than to estimate full binary classification performance.

In [94]:
split_date = pd.Timestamp("2003-11-23")

train_df = features_df[
    features_df["timestamp"] < split_date
].copy()

test_df = features_df[
    features_df["timestamp"] >= split_date
].copy()

In [95]:
print("Training samples:", len(train_df))
print("Test samples:", len(test_df))

print("\nTraining classes:")
print(train_df["target"].value_counts())

print("\nTest classes:")
print(test_df["target"].value_counts())

Training samples: 1862
Test samples: 294

Training classes:
target
0    1507
1     355
Name: count, dtype: int64

Test classes:
target
1    294
Name: count, dtype: int64


## Predictor and target matrices

The engineered vibration features are used as model inputs, while the heuristic condition label is used as the supervised target.

The timestamp is excluded from the predictor matrix because the goal is to classify bearing condition from vibration behavior rather than directly from calendar time.

In [96]:
ml_features = [
    "rms",
    "std",
    "peak",
    "crest_factor",
    "kurtosis",
    "dominant_frequency",
    "spectral_energy",
]

X_train = train_df[ml_features]
y_train = train_df["target"]

X_test = test_df[ml_features]
y_test = test_df["target"]

In [97]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (1862, 7)
y_train: (1862,)
X_test: (294, 7)
y_test: (294,)


In [98]:
print("Training target:")
print(y_train.value_counts())

print("\nTest target:")
print(y_test.value_counts())

Training target:
target
0    1507
1     355
Name: count, dtype: int64

Test target:
target
1    294
Name: count, dtype: int64


## Feature standardization for Logistic Regression

The engineered features have different numerical scales.

For example, RMS and kurtosis have relatively small values, while spectral energy can be much larger.

Logistic Regression estimates coefficients from these predictors, so the features are standardized before training.

The standardization is:

$$
z = \frac{x - \mu}{\sigma}
$$

where $\mu$ and $\sigma$ are calculated using the training data only.

The scaler is fitted only on `X_train` to prevent information from the future test period from leaking into the training process.

In [99]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [100]:
print("Training scaled shape:", X_train_scaled.shape)
print("Test scaled shape:", X_test_scaled.shape)

print("\nTraining feature means:")
print(X_train_scaled.mean(axis=0))

print("\nTraining feature standard deviations:")
print(X_train_scaled.std(axis=0))

Training scaled shape: (1862, 7)
Test scaled shape: (294, 7)

Training feature means:
[-2.44225215e-16 -6.10563038e-17  6.10563038e-17 -1.83168911e-16
  1.22112608e-16  6.10563038e-17  1.52640760e-16]

Training feature standard deviations:
[1. 1. 1. 1. 1. 1. 1.]


## Logistic Regression baseline

Logistic Regression is used as the first supervised baseline.

It estimates the probability that a snapshot belongs to the DEGRADED class based on the engineered vibration features.

The model is intentionally simple and interpretable, making it useful as a reference point before testing more flexible nonlinear models.

The model is trained only on the chronological training period and evaluated on the final unseen period of the experiment.

In [101]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

log_reg.fit(
    X_train_scaled,
    y_train,
)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

In [102]:
y_pred_log = log_reg.predict(X_test_scaled)

y_prob_log = log_reg.predict_proba(X_test_scaled)[:, 1]

In [103]:
pd.Series(y_pred_log).value_counts()

1    294
Name: count, dtype: int64

In [104]:
y_prob_log[:10]

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [105]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    confusion_matrix,
)

test_accuracy = accuracy_score(y_test, y_pred_log)
test_recall = recall_score(y_test, y_pred_log)

print("Test accuracy:", test_accuracy)
print("Degraded recall:", test_recall)

print("\nConfusion matrix:")
print(confusion_matrix(
    y_test,
    y_pred_log,
    labels=[0, 1],
))

Test accuracy: 1.0
Degraded recall: 1.0

Confusion matrix:
[[  0   0]
 [  0 294]]


## Grouped cross-validation

The chronological holdout evaluates generalization to the final degraded period, but it contains no NORMAL observations.

To obtain a complementary evaluation of binary classification performance, grouped cross-validation is used.

Snapshots are grouped by recording date so that observations from the same day are kept together. Entire days are assigned to either training or validation within each fold.

Stratification is used so that each validation fold contains observations from both NORMAL and DEGRADED periods.

This evaluation is not strictly chronological because training folds may contain dates later than some validation folds. Therefore, it is used only to assess class separability and complements, rather than replaces, the chronological holdout.

In [106]:
groups = features_df["timestamp"].dt.date

X = features_df[ml_features]
y = features_df["target"]

In [107]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

In [108]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [109]:
log_reg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42,
        ),
    ),
])

In [110]:
for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y, groups=groups),
    start=1,
):
    y_val = y.iloc[val_idx]

    print(f"Fold {fold}")
    print(y_val.value_counts().sort_index())
    print()

Fold 1
target
0    298
1    117
Name: count, dtype: int64

Fold 2
target
0    289
1    147
Name: count, dtype: int64

Fold 3
target
0    312
1    126
Name: count, dtype: int64

Fold 4
target
0    309
1    131
Name: count, dtype: int64

Fold 5
target
0    299
1    128
Name: count, dtype: int64



## Logistic Regression cross-validation

The Logistic Regression pipeline is evaluated using the grouped folds defined above.

For each fold:

- the model is trained on the training days,
- the scaler is fitted only on the training portion,
- the model is evaluated on held-out days,
- accuracy, precision, recall, and F1-score are recorded.

The final result is summarized using the mean and standard deviation across the five folds.

In [111]:
from sklearn.model_selection import cross_validate

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

cv_results_log = cross_validate(
    log_reg_pipeline,
    X,
    y,
    groups=groups,
    cv=cv,
    scoring=scoring,
)

In [112]:
cv_results_log

{'fit_time': array([0.00265694, 0.00207901, 0.00253201, 0.00235415, 0.00226974]),
 'score_time': array([0.00313878, 0.0036552 , 0.00341892, 0.00297761, 0.003901  ]),
 'test_accuracy': array([0.84096386, 0.75688073, 0.99771689, 1.        , 1.        ]),
 'test_precision': array([0.63934426, 1.        , 0.99212598, 1.        , 1.        ]),
 'test_recall': array([1.        , 0.27891156, 1.        , 1.        , 1.        ]),
 'test_f1': array([0.78      , 0.43617021, 0.99604743, 1.        , 1.        ])}

In [113]:
for metric in ["accuracy", "precision", "recall", "f1"]:
    scores = cv_results_log[f"test_{metric}"]

    print(
        f"{metric.capitalize()}: "
        f"{scores.mean():.3f} ± {scores.std():.3f}"
    )

Accuracy: 0.919 ± 0.102
Precision: 0.926 ± 0.144
Recall: 0.856 ± 0.288
F1: 0.842 ± 0.220


## Random Forest classifier

Random Forest is used as a stronger nonlinear supervised model.

Unlike Logistic Regression, Random Forest can model nonlinear relationships and interactions between vibration features without requiring them to be specified manually.

It builds multiple decision trees and combines their predictions to improve robustness and reduce overfitting compared with a single decision tree.

Feature standardization is not required because decision-tree splits depend on feature thresholds rather than feature magnitude.

The model is evaluated using the same grouped cross-validation strategy used for Logistic Regression so that the results can be compared fairly.

In [114]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)

In [115]:
cv_results_rf = cross_validate(
    random_forest,
    X,
    y,
    groups=groups,
    cv=cv,
    scoring=scoring,
)

In [116]:
for metric in ["accuracy", "precision", "recall", "f1"]:
    scores = cv_results_rf[f"test_{metric}"]

    print(
        f"{metric.capitalize()}: "
        f"{scores.mean():.3f} ± {scores.std():.3f}"
    )

Accuracy: 0.930 ± 0.093
Precision: 0.939 ± 0.121
Recall: 0.864 ± 0.272
F1: 0.861 ± 0.201


In [117]:
cv_results_rf["test_recall"]

array([1.        , 0.31972789, 1.        , 1.        , 1.        ])

## Investigating the weak validation fold

Both Logistic Regression and Random Forest show substantially lower recall in Fold 2.

To understand whether this is caused by a specific stage of degradation, the dates assigned to Fold 2 are inspected together with their class labels.

This helps determine whether the weak performance is associated with the transition into degradation rather than with random model instability.

In [118]:
fold_2_train_idx, fold_2_val_idx = list(
    cv.split(X, y, groups=groups)
)[1]

fold_2_df = features_df.iloc[fold_2_val_idx].copy()

In [119]:
fold_2_summary = (
    fold_2_df
    .groupby(fold_2_df["timestamp"].dt.date)["target"]
    .agg(["count", "sum"])
)

fold_2_summary

,count,sum
timestamp,,
2003-10-23,62,0
2003-11-01,144,0
2003-11-17,83,0
2003-11-20,110,110
2003-11-25,37,37


### Fold 2 performance by degraded date

Fold 2 contains degraded observations from November 20 and November 25.

Because November 20 is the first day of the heuristic degraded period, it may represent a transitional condition that is harder to distinguish from normal operation.

The model predictions are therefore examined separately for each degraded date.

In [120]:
X_fold2_train = X.iloc[fold_2_train_idx]
y_fold2_train = y.iloc[fold_2_train_idx]

X_fold2_val = X.iloc[fold_2_val_idx]
y_fold2_val = y.iloc[fold_2_val_idx]

random_forest.fit(
    X_fold2_train,
    y_fold2_train,
)

fold_2_df["rf_prediction"] = random_forest.predict(
    X_fold2_val
)

In [121]:
fold_2_degraded_summary = (
    fold_2_df[fold_2_df["target"] == 1]
    .groupby(fold_2_df["timestamp"].dt.date)
    .agg(
        samples=("target", "size"),
        detected=("rf_prediction", "sum"),
    )
)

fold_2_degraded_summary["recall"] = (
    fold_2_degraded_summary["detected"]
    / fold_2_degraded_summary["samples"]
)

fold_2_degraded_summary

,samples,detected,recall
timestamp,,,
2003-11-20,110,10,0.090909
2003-11-25,37,37,1.000000


### Interpretation of the weak fold

The low recall observed in Fold 2 is primarily caused by November 20, the first day assigned to the heuristic DEGRADED class.

Random Forest detects only 10 of the 110 November 20 snapshots as degraded, corresponding to a recall of approximately 9.1%.

In contrast, all 37 snapshots from November 25 are correctly detected as degraded.

This suggests that the weak validation performance is not caused by a general inability to recognize degradation. Instead, November 20 appears to represent a transitional operating regime whose vibration characteristics still overlap substantially with the earlier NORMAL condition.

The result highlights an important limitation of the binary heuristic labeling strategy: degradation is a gradual physical process, while the target changes abruptly from NORMAL to DEGRADED at a fixed timestamp.

## Model comparison

Three baseline approaches were evaluated:

1. **Isolation Forest**
   - trained only on early healthy behavior,
   - produces an anomaly score,
   - does not require condition labels during training.

2. **Logistic Regression**
   - supervised linear classifier,
   - trained using the heuristic NORMAL/DEGRADED labels,
   - provides a simple and interpretable baseline.

3. **Random Forest**
   - supervised nonlinear classifier,
   - captures more complex relationships between vibration features,
   - provides a stronger comparison against Logistic Regression.

The supervised models are compared using grouped cross-validation, while the Isolation Forest is evaluated against the heuristic labels as a consistency check.

In [123]:
comparison_df = pd.DataFrame({
    "model": [
        "Isolation Forest",
        "Logistic Regression",
        "Random Forest",
    ],
    "accuracy": [
        0.918,
        0.919,
        0.930,
    ],
    "precision": [
        0.786,
        0.926,
        0.939,
    ],
    "recall": [
        1.000,
        0.856,
        0.864,
    ],
    "f1": [
        0.880,
        0.842,
        0.861,
    ],
})

comparison_df

,model,accuracy,precision,recall,f1
0,Isolation Forest,0.918,0.786,1.000,0.880
1,Logistic Regression,0.919,0.926,0.856,0.842
2,Random Forest,0.930,0.939,0.864,0.861


The metrics above should not be interpreted as a strict ranking because the evaluation procedures differ.

Isolation Forest was evaluated against the heuristic labels across the full experiment, while the supervised models were evaluated using grouped cross-validation.

The comparison is therefore used to understand the strengths and weaknesses of each modeling approach rather than to select a model based only on the highest metric.

## Baseline modeling conclusions

The baseline experiments show that the engineered vibration features contain substantial information about bearing condition.

Both Logistic Regression and Random Forest achieve strong overall classification performance, with Random Forest providing a small improvement over the linear baseline.

However, both supervised models show reduced recall around the beginning of the heuristic degraded period. Detailed analysis of the weak validation fold shows that November 20 behaves as a transitional regime rather than as a clearly degraded state.

This highlights a limitation of using a fixed binary degradation boundary for a gradual physical process.

For the first MachineGuard pipeline, the Isolation Forest anomaly score is therefore retained as the primary condition-monitoring signal. It can be trained using healthy observations only and provides a continuous measure of deviation from the healthy baseline.

The supervised models remain useful as reference models and demonstrate that the extracted vibration features can distinguish strongly degraded conditions from normal operation.

Future stages can transform the anomaly score into higher-level health states such as NORMAL, DEGRADING, and CRITICAL.